# 15 - Polars: introducción — **Soluciones**

In [ ]:

import numpy as np, pandas as pd
try:
    import polars as pl; HAS_POLARS=True; print("Polars OK")
except Exception as e:
    HAS_POLARS=False; print("Polars no disponible:", e)
rng = np.random.default_rng(0)
n=1000
pdf = pd.DataFrame({
    "id": np.arange(n),
    "edad": rng.integers(18,70,size=n),
    "ingresos": rng.normal(26000,9000,size=n).round(2),
    "ciudad": rng.choice(["Cádiz","Sevilla","Málaga","Córdoba"], size=n),
    "compra": rng.choice([0,1], size=n, p=[0.45,0.55])
})
if HAS_POLARS: df = pl.from_pandas(pdf)


In [ ]:

if HAS_POLARS:
    subset = df.select(["edad","ingresos","compra"]).head(3)
    filtrado = df.filter(pl.col("ingresos")>30000).head(3)
    ordenado = df.sort(["ciudad","ingresos"], descending=[False,True]).head(3)
else:
    subset = pdf[["edad","ingresos","compra"]].head(3)
    filtrado = pdf[pdf["ingresos"]>30000].head(3)
    ordenado = pdf.sort_values(["ciudad","ingresos"], ascending=[True,False]).head(3)
subset, filtrado, ordenado


In [ ]:

if HAS_POLARS:
    res = (df.lazy()
           .filter(pl.col("edad")>30)
           .with_columns((pl.col("ingresos")/1000).alias("ingresos_k"))
           .groupby("ciudad").agg(pl.col("ingresos_k").mean().alias("media_k"))
           .sort("media_k", descending=True)
           .collect())
else:
    res = (pdf.assign(ingresos_k = pdf["ingresos"]/1000)
             .query("edad>30")
             .groupby("ciudad")["ingresos_k"].mean()
             .sort_values(ascending=False).reset_index(name="media_k"))
res.head()



---

## Tests automáticos


In [ ]:

import numpy as np, pandas as pd
try:
    import polars as pl; HAS_POLARS=True
except: HAS_POLARS=False
rng=np.random.default_rng(0); n=1000
pdf=pd.DataFrame({
    "id": np.arange(n),
    "edad": rng.integers(18,70,size=n),
    "ingresos": rng.normal(26000,9000,size=n).round(2),
    "ciudad": rng.choice(["Cádiz","Sevilla","Málaga","Córdoba"], size=n),
    "compra": rng.choice([0,1], size=n, p=[0.45,0.55])
})
if HAS_POLARS:
    df=pl.from_pandas(pdf)
    res=(df.lazy().filter(pl.col("edad")>30)
         .with_columns((pl.col("ingresos")/1000).alias("ingresos_k"))
         .groupby("ciudad").agg(pl.col("ingresos_k").mean().alias("media_k"))
         .sort("media_k", descending=True).collect())
else:
    res=(pdf.assign(ingresos_k=pdf["ingresos"]/1000).query("edad>30")
         .groupby("ciudad")["ingresos_k"].mean().sort_values(ascending=False)
         .reset_index(name="media_k"))


In [ ]:

def check(c,o,f):
    if c: print("✔",o)
    else: raise AssertionError("✘ "+f)

check(len(pdf)==1000, "Dataset con 1000 filas.", "Tamaño de dataset incorrecto.")
cols = res.columns if not HAS_POLARS else res.columns
check(("media_k" in cols), "Columna 'media_k' presente.", "Falta 'media_k' en el resultado.")
